# Inverse Positioning From Magnetic Trajectories

This notebook builds a smooth synthetic magnetic-like field on sphere coordinates `x = lon`, `y = sin(lat)`, simulates noisy biased-direction random-walk measurement trajectories, and estimates `H(start | trajectory)` and `I(start; trajectory)` from a discrete uniform start-position prior.

## 1. Preparations

In [1]:
import math
from dataclasses import dataclass

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import logsumexp
from tqdm.auto import tqdm

SEED = 42

# Noise controls. NOISE is the absolute Gaussian std used for augmented field
# measurements and for the trajectory observation model.
NOISE = 1e-1

# Trajectory experiment controls.
STEP_SIZE_CHOICES = np.array([0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1], dtype=float)
STEP_COUNT_CHOICES = np.array([1, 2, 4, 8, 12, 16], dtype=int)
EXPERIMENT_REPEATS = 10

# Biased-direction random walk controls.
DIRECTION_HEADING = 0.0
DIRECTION_MAGNITUDE = 0.75
DIRECTION_MAX_KAPPA = 12.0

# Posterior approximation controls.
GRID_LON_BINS = 128
GRID_Y_BINS = int(math.pi * GRID_LON_BINS // 2)
POSTERIOR_BATCH = 512

# Smooth internal magnetic dipoles. Sources stay inside the sphere so the field
# is smooth on the surface and periodic in longitude.
FIELD_SOURCE_POSITIONS = np.array(
    [
        [0.00, 0.00, 0.00],
        [0.32, -0.08, 0.10],
        [-0.18, 0.26, -0.22],
        [0.09, 0.05, 0.34],
    ],
    dtype=float,
)
FIELD_SOURCE_MOMENTS = np.array(
    [
        [0.20, -0.10, 1.00],
        [-0.70, 0.40, 0.30],
        [0.50, 0.80, -0.20],
        [0.30, -0.60, 0.50],
    ],
    dtype=float,
)
FIELD_SOURCE_WEIGHTS = np.array([1.00, 0.22, 0.18, 0.14], dtype=float)
FIELD_EPS = 1e-12

# Smooth fractal detail controls. These octave plane waves are evaluated in
# 3D Cartesian coordinates and then restricted to the sphere, so they remain
# seamless in longitude and well-defined at the poles.
FIELD_DETAIL_SEED = 2025
FIELD_DETAIL_OCTAVES = 6
FIELD_DETAIL_BASIS_PER_OCTAVE = 12
FIELD_DETAIL_BASE_FREQUENCY = 1.5
FIELD_DETAIL_LACUNARITY = 2.0
FIELD_DETAIL_PERSISTENCE = 0.55
FIELD_DETAIL_AMPLITUDE = 0.22

field_detail_rng = np.random.default_rng(FIELD_DETAIL_SEED)
FIELD_DETAIL_DIRECTIONS = field_detail_rng.normal(
    size=(FIELD_DETAIL_OCTAVES, FIELD_DETAIL_BASIS_PER_OCTAVE, 3)
)
FIELD_DETAIL_DIRECTIONS /= np.linalg.norm(
    FIELD_DETAIL_DIRECTIONS, axis=-1, keepdims=True
)
FIELD_DETAIL_SIN_VECTORS = field_detail_rng.normal(
    size=(FIELD_DETAIL_OCTAVES, FIELD_DETAIL_BASIS_PER_OCTAVE, 3)
)
FIELD_DETAIL_COS_VECTORS = field_detail_rng.normal(
    size=(FIELD_DETAIL_OCTAVES, FIELD_DETAIL_BASIS_PER_OCTAVE, 3)
)
FIELD_DETAIL_PHASES = field_detail_rng.uniform(
    0.0, 2.0 * np.pi, size=(FIELD_DETAIL_OCTAVES, FIELD_DETAIL_BASIS_PER_OCTAVE)
)
FIELD_DETAIL_FREQUENCIES = (
    FIELD_DETAIL_BASE_FREQUENCY
    * FIELD_DETAIL_LACUNARITY ** np.arange(FIELD_DETAIL_OCTAVES)
)
FIELD_DETAIL_AMPLITUDES = (
    FIELD_DETAIL_AMPLITUDE * FIELD_DETAIL_PERSISTENCE ** np.arange(FIELD_DETAIL_OCTAVES)
)

rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

c:\Users\shich\Src\inverse_positioning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Shared Functional

Sphere geometry, smooth field generation, dataset creation, trajectory stepping, posterior entropy estimation, and Plotly helpers shared by all experiments.

### Smooth sphere field

The field is evaluated from dipole sources plus smooth octave detail in 3D Cartesian coordinates and then sampled on the unit sphere. This makes longitude wrapping exact because `x = -pi` and `x = pi` map to the same points. It also avoids singular sources on the surface. The fine detail is fractal-style noise built from plane waves with increasing frequency and decreasing amplitude.

In [2]:
def wrap_lon(lon):
    """Wrap longitude to [-pi, pi)."""
    return (np.asarray(lon) + np.pi) % (2.0 * np.pi) - np.pi


def sphere_xyz(lon, y):
    """Convert lon and y=sin(lat) to unit-sphere Cartesian coordinates."""
    lon, y = np.broadcast_arrays(
        np.asarray(lon, dtype=float), np.asarray(y, dtype=float)
    )
    y = np.clip(y, -1.0, 1.0)
    rho = np.sqrt(np.maximum(0.0, 1.0 - y * y))
    return np.stack((rho * np.cos(lon), rho * np.sin(lon), y), axis=-1)


def lon_y_from_xyz(xyz):
    """Convert unit-sphere Cartesian coordinates back to lon and y=sin(lat)."""
    xyz = np.asarray(xyz, dtype=float)
    lon = np.arctan2(xyz[..., 1], xyz[..., 0])
    y = np.clip(xyz[..., 2], -1.0, 1.0)
    return wrap_lon(lon), y


def sphere_fractal_detail(lon, y):
    """Smooth vector-valued octave noise restricted to the unit sphere."""
    r = sphere_xyz(lon, y)
    detail = np.zeros_like(r)

    for frequency, amplitude, directions, sin_vectors, cos_vectors, phases in zip(
        FIELD_DETAIL_FREQUENCIES,
        FIELD_DETAIL_AMPLITUDES,
        FIELD_DETAIL_DIRECTIONS,
        FIELD_DETAIL_SIN_VECTORS,
        FIELD_DETAIL_COS_VECTORS,
        FIELD_DETAIL_PHASES,
    ):
        argument = frequency * np.einsum("...d,bd->...b", r, directions) + phases
        octave = np.einsum("...b,bd->...d", np.sin(argument), sin_vectors) + np.einsum(
            "...b,bd->...d", np.cos(argument), cos_vectors
        )
        detail += amplitude * octave / math.sqrt(FIELD_DETAIL_BASIS_PER_OCTAVE)

    return detail


def magnetic_field(lon, y):
    """Magnetic-like vector field B(lon, y) = (Bx, By, Bz)."""
    r = sphere_xyz(lon, y)
    field = np.zeros_like(r)

    for source_pos, moment, weight in zip(
        FIELD_SOURCE_POSITIONS, FIELD_SOURCE_MOMENTS, FIELD_SOURCE_WEIGHTS
    ):
        R = r - source_pos
        radius2 = np.sum(R * R, axis=-1, keepdims=True)
        radius = np.sqrt(np.maximum(radius2, FIELD_EPS))
        moment_dot_R = np.sum(moment * R, axis=-1, keepdims=True)
        field += weight * (3.0 * R * moment_dot_R / radius**5 - moment / radius**3)

    return field + sphere_fractal_detail(lon, y)


def continuity_checks():
    y_probe = np.linspace(-1.0, 1.0, 1_001)
    x_probe = np.linspace(-np.pi, np.pi, 721)
    h = 1e-5

    seam_value_gap = np.max(
        np.abs(magnetic_field(-np.pi, y_probe) - magnetic_field(np.pi, y_probe))
    )
    ddx_left = (
        magnetic_field(-np.pi + h, y_probe) - magnetic_field(-np.pi - h, y_probe)
    ) / (2.0 * h)
    ddx_right = (
        magnetic_field(np.pi + h, y_probe) - magnetic_field(np.pi - h, y_probe)
    ) / (2.0 * h)
    seam_differential_gap = np.max(np.abs(ddx_left - ddx_right))

    north_values = magnetic_field(x_probe, np.ones_like(x_probe))
    south_values = magnetic_field(x_probe, -np.ones_like(x_probe))
    north_d_dx = (
        magnetic_field(x_probe + h, 1.0) - magnetic_field(x_probe - h, 1.0)
    ) / (2.0 * h)
    south_d_dx = (
        magnetic_field(x_probe + h, -1.0) - magnetic_field(x_probe - h, -1.0)
    ) / (2.0 * h)

    return {
        "longitude seam value gap": seam_value_gap,
        "longitude seam d/dx gap": seam_differential_gap,
        "north pole longitude variation": np.ptp(north_values, axis=0).max(),
        "south pole longitude variation": np.ptp(south_values, axis=0).max(),
        "north pole max |dB/dx|": np.linalg.norm(north_d_dx, axis=-1).max(),
        "south pole max |dB/dx|": np.linalg.norm(south_d_dx, axis=-1).max(),
    }


for name, value in continuity_checks().items():
    print(f"{name:34s}: {value:.3e}")

longitude seam value gap          : 8.882e-16
longitude seam d/dx gap           : 6.661e-11
north pole longitude variation    : 0.000e+00
south pole longitude variation    : 0.000e+00
north pole max |dB/dx|            : 0.000e+00
south pole max |dB/dx|            : 0.000e+00


In [3]:
grid_x = np.linspace(-np.pi, np.pi, 241)
grid_y = np.linspace(-1.0, 1.0, 121)
Xg, Yg = np.meshgrid(grid_x, grid_y)
Bg = magnetic_field(Xg, Yg)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=["Bx", "By", "Bz"],
    horizontal_spacing=0.12,
)

colorbar_x = [0.275, 0.635, 0.995]
for col, component in enumerate(["Bx", "By", "Bz"], start=1):
    fig.add_trace(
        go.Heatmap(
            x=grid_x,
            y=grid_y,
            z=Bg[..., col - 1],
            colorscale="RdBu",
            reversescale=True,
            colorbar={
                "title": component,
                "len": 0.78,
                "thickness": 14,
                "x": colorbar_x[col - 1],
            },
            hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<br>"
            + component
            + "=%{z:.3f}<extra></extra>",
        ),
        row=1,
        col=col,
    )
    fig.update_xaxes(title_text="x = lon", row=1, col=col, constrain="domain")
    fig.update_yaxes(
        title_text="y = sin(lat)",
        row=1,
        col=col,
        scaleanchor=f"x{col}" if col > 1 else "x",
        scaleratio=np.pi,
        title_standoff=8,
    )

fig.update_layout(
    title={"text": "smooth magnetic field components", "x": 0.5},
    height=440,
    width=1320,
    margin={"l": 60, "r": 90, "t": 70, "b": 70},
)
fig.show()

### Shared trajectory and entropy utilities

In [4]:
def local_east_north(lon, y):
    lon, y = np.broadcast_arrays(
        np.asarray(lon, dtype=float), np.asarray(y, dtype=float)
    )
    rho = np.sqrt(np.maximum(0.0, 1.0 - np.clip(y, -1.0, 1.0) ** 2))
    east = np.stack((-np.sin(lon), np.cos(lon), np.zeros_like(lon)), axis=-1)
    north = np.stack((-y * np.cos(lon), -y * np.sin(lon), rho), axis=-1)
    return east, north


def geodesic_step(lon, y, step_size, heading):
    r = sphere_xyz(lon, y)
    east, north = local_east_north(lon, y)
    tangent = np.cos(heading) * east + np.sin(heading) * north
    r_next = np.cos(step_size) * r + np.sin(step_size) * tangent
    r_next = r_next / np.linalg.norm(r_next, axis=-1, keepdims=True)
    return lon_y_from_xyz(r_next)


def apply_walk(start_lon, start_y, step_size, headings):
    lon = np.asarray(start_lon, dtype=float)
    y = np.asarray(start_y, dtype=float)
    lons = [lon]
    ys = [y]

    for heading in headings:
        lon, y = geodesic_step(lon, y, step_size, heading)
        lons.append(lon)
        ys.append(y)

    return np.stack(lons, axis=-1), np.stack(ys, axis=-1)


@dataclass(frozen=True)
class Trajectory:
    repeat: int
    start_lon: float
    start_y: float
    step_size: float
    requested_step_count: int
    headings: np.ndarray
    positions_lon: np.ndarray
    positions_y: np.ndarray
    clean_B: np.ndarray
    observed_B: np.ndarray

    @property
    def step_count(self):
        return len(self.headings)


def make_headings(rng, step_count):
    kappa = DIRECTION_MAGNITUDE * DIRECTION_MAX_KAPPA
    return rng.vonmises(DIRECTION_HEADING, kappa, size=step_count)


def make_trajectory(rng, step_size, step_count, repeat):
    start_lon = rng.uniform(-np.pi, np.pi)
    start_y = rng.uniform(-1.0, 1.0)
    headings = make_headings(rng, step_count)

    positions_lon, positions_y = apply_walk(start_lon, start_y, step_size, headings)
    clean_B = magnetic_field(positions_lon, positions_y)
    observed_B = clean_B + rng.normal(0.0, NOISE, size=clean_B.shape)

    return Trajectory(
        repeat=repeat,
        start_lon=start_lon,
        start_y=start_y,
        step_size=step_size,
        requested_step_count=step_count,
        headings=headings,
        positions_lon=positions_lon,
        positions_y=positions_y,
        clean_B=clean_B,
        observed_B=observed_B,
    )


def make_trajectories(rng):
    return [
        make_trajectory(rng, float(step_size), int(step_count), repeat)
        for step_size in STEP_SIZE_CHOICES
        for step_count in STEP_COUNT_CHOICES
        for repeat in range(EXPERIMENT_REPEATS)
    ]

### Conditional entropy and mutual information

The starting position prior is a uniform grid over `x` and `y`, which is uniform area on the sphere. The posterior for one trajectory is proportional to the Gaussian likelihood of the observed magnetic sequence under each candidate start and the known walk controls.

In [5]:
def candidate_grid(n_lon, n_y):
    dx = 2.0 * np.pi / n_lon
    dy = 2.0 / n_y
    lon_centers = np.linspace(-np.pi + 0.5 * dx, np.pi - 0.5 * dx, n_lon)
    y_centers = np.linspace(-1.0 + 0.5 * dy, 1.0 - 0.5 * dy, n_y)
    L, Y = np.meshgrid(lon_centers, y_centers, indexing="xy")
    return L.ravel(), Y.ravel()


def angular_distance(lon1, y1, lon2, y2):
    r1 = sphere_xyz(lon1, y1)
    r2 = sphere_xyz(lon2, y2)
    cosine = np.sum(r1 * r2, axis=-1)
    return np.arccos(np.clip(cosine, -1.0, 1.0))


CANDIDATE_LON, CANDIDATE_Y = candidate_grid(GRID_LON_BINS, GRID_Y_BINS)
CANDIDATES = CANDIDATE_LON.size
PRIOR_ENTROPY_NATS = math.log(CANDIDATES)
PRIOR_ENTROPY_BITS = PRIOR_ENTROPY_NATS / math.log(2.0)

print(f"candidate starts: {CANDIDATES:,}")
print(
    f"discrete prior entropy: {PRIOR_ENTROPY_NATS:.3f} nats = {PRIOR_ENTROPY_BITS:.3f} bits"
)

candidate starts: 25,728
discrete prior entropy: 10.155 nats = 14.651 bits


In [6]:
def trajectory_log_likelihood(trajectory):
    log_likelihood = np.empty(CANDIDATES, dtype=float)

    for start in range(0, CANDIDATES, POSTERIOR_BATCH):
        stop = min(start + POSTERIOR_BATCH, CANDIDATES)
        cand_lon = CANDIDATE_LON[start:stop]
        cand_y = CANDIDATE_Y[start:stop]

        pred_lon, pred_y = apply_walk(
            cand_lon, cand_y, trajectory.step_size, trajectory.headings
        )
        pred_B = magnetic_field(pred_lon, pred_y)
        residual = pred_B - trajectory.observed_B[np.newaxis, :, :]
        log_likelihood[start:stop] = (
            -0.5 * np.sum(residual * residual, axis=(1, 2)) / (NOISE * NOISE)
        )

    return log_likelihood


def posterior_for_trajectory(trajectory):
    log_likelihood = trajectory_log_likelihood(trajectory)
    log_posterior = log_likelihood - logsumexp(log_likelihood)
    posterior = np.exp(log_posterior)
    entropy_nats = -np.sum(posterior * log_posterior)
    mi_nats = PRIOR_ENTROPY_NATS - entropy_nats
    map_index = int(np.argmax(posterior))
    map_lon = CANDIDATE_LON[map_index]
    map_y = CANDIDATE_Y[map_index]

    return {
        "posterior": posterior,
        "entropy_nats": entropy_nats,
        "entropy_bits": entropy_nats / math.log(2.0),
        "mi_nats": mi_nats,
        "mi_bits": mi_nats / math.log(2.0),
        "map_lon": map_lon,
        "map_y": map_y,
        "map_error_rad": float(
            angular_distance(trajectory.start_lon, trajectory.start_y, map_lon, map_y)
        ),
        "posterior_perplexity": float(np.exp(entropy_nats)),
    }


def estimate_information(trajectories, progress_label="trajectory"):
    rows = []
    iterator = tqdm(
        enumerate(trajectories),
        total=len(trajectories),
        desc=f"estimating {progress_label}",
        unit="traj",
    )
    for index, trajectory in iterator:
        posterior = posterior_for_trajectory(trajectory)
        rows.append(
            {
                "trajectory": index,
                "repeat": trajectory.repeat,
                "step_size": trajectory.step_size,
                "steps": trajectory.step_count,
                "conditional_entropy_nats": posterior["entropy_nats"],
                "conditional_entropy_bits": posterior["entropy_bits"],
                "mutual_information_nats": posterior["mi_nats"],
                "mutual_information_bits": posterior["mi_bits"],
                "posterior_perplexity": posterior["posterior_perplexity"],
                "map_error_rad": posterior["map_error_rad"],
            }
        )
    return rows


def summarize_values(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.nan, 0.0, 0
    sem = values.std(ddof=1) / np.sqrt(values.size) if values.size > 1 else 0.0
    return values.mean(), sem, values.size


def rows_to_arrays(rows):
    return {
        "conditional_bits": np.array([row["conditional_entropy_bits"] for row in rows]),
        "mi_bits": np.array([row["mutual_information_bits"] for row in rows]),
        "map_errors": np.array([row["map_error_rad"] for row in rows]),
    }


def print_information_summary(rows, title):
    arrays = rows_to_arrays(rows)
    print(f"\n{title}")
    print(f"prior entropy:                 {PRIOR_ENTROPY_BITS:8.3f} bits")
    print(
        f"mean H(start | trajectory):    {arrays['conditional_bits'].mean():8.3f} bits"
    )
    print(
        f"median H(start | trajectory):  {np.median(arrays['conditional_bits']):8.3f} bits"
    )
    print(f"mean I(start; trajectory):     {arrays['mi_bits'].mean():8.3f} bits")
    print(f"median I(start; trajectory):   {np.median(arrays['mi_bits']):8.3f} bits")
    print(f"median MAP angular error:      {np.median(arrays['map_errors']):8.4f} rad")


def mi_values(rows, predicate):
    return [row["mutual_information_bits"] for row in rows if predicate(row)]


def grouped_mean_sem(rows, group_key, groups):
    output = []
    for group in groups:
        values = [
            row["mutual_information_bits"] for row in rows if row[group_key] == group
        ]
        mean, sem, count = summarize_values(values)
        output.append({"group": group, "mean": mean, "sem": sem, "count": count})
    return output


def add_entropy_line(fig, row=None, col=None, label="H(start)"):
    fig.add_hline(
        y=PRIOR_ENTROPY_BITS,
        line_color="red",
        line_dash="dash",
        line_width=1.2,
        annotation_text=label,
        annotation_position="top left",
        row=row,
        col=col,
    )


def plot_aggregate_mi(rows, title_prefix):
    by_step_size = grouped_mean_sem(rows, "step_size", STEP_SIZE_CHOICES)
    by_step_count = grouped_mean_sem(rows, "steps", STEP_COUNT_CHOICES)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            f"{title_prefix}: E[I(start; trajectory)] by step size",
            f"{title_prefix}: E[I(start; trajectory)] by steps count",
        ],
    )
    fig.add_trace(
        go.Scatter(
            x=[r["group"] for r in by_step_size],
            y=[r["mean"] for r in by_step_size],
            error_y={
                "type": "data",
                "array": [r["sem"] for r in by_step_size],
                "visible": True,
            },
            mode="lines+markers+text",
            text=[f"n={r['count']}" for r in by_step_size],
            textposition="top center",
            name="step size",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[r["group"] for r in by_step_count],
            y=[r["mean"] for r in by_step_count],
            error_y={
                "type": "data",
                "array": [r["sem"] for r in by_step_count],
                "visible": True,
            },
            mode="lines+markers+text",
            text=[f"n={r['count']}" for r in by_step_count],
            textposition="top center",
            name="steps count",
        ),
        row=1,
        col=2,
    )
    add_entropy_line(fig, row=1, col=1)
    add_entropy_line(fig, row=1, col=2)
    fig.update_xaxes(title_text="step size", type="log", row=1, col=1)
    fig.update_xaxes(title_text="steps count", row=1, col=2)
    fig.update_yaxes(title_text="expected mutual information [bits]", row=1, col=1)
    fig.update_yaxes(title_text="expected mutual information [bits]", row=1, col=2)
    fig.update_layout(height=450, width=1150, showlegend=False)
    fig.show()


def plot_mi_heatmap(rows, title):
    matrix = np.full((len(STEP_COUNT_CHOICES), len(STEP_SIZE_CHOICES)), np.nan)
    for i, step_count in enumerate(STEP_COUNT_CHOICES):
        for j, step_size in enumerate(STEP_SIZE_CHOICES):
            values = mi_values(
                rows,
                lambda r, sc=step_count, ss=step_size: r["steps"] == sc
                and r["step_size"] == ss,
            )
            matrix[i, j] = np.mean(values) if values else np.nan

    fig = go.Figure(
        data=go.Heatmap(
            x=[f"{v:g}" for v in STEP_SIZE_CHOICES],
            y=[str(v) for v in STEP_COUNT_CHOICES],
            z=matrix,
            colorscale="Viridis",
            colorbar={"title": "MI [bits]"},
            zmin=np.nanmin(matrix),
            zmax=PRIOR_ENTROPY_BITS,
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="step size",
        yaxis_title="steps count",
        height=450,
        width=650,
    )
    fig.show()


def wrap_aware_segments(lons, ys, wrap_threshold=np.pi):
    lons = np.asarray(lons, dtype=float)
    ys = np.asarray(ys, dtype=float)
    segments = []
    start = 0
    jumps = np.where(np.abs(np.diff(lons)) > wrap_threshold)[0]
    for jump in jumps:
        stop = jump + 1
        segments.append((lons[start:stop], ys[start:stop]))
        start = stop
    segments.append((lons[start:], ys[start:]))
    return segments


def plot_trajectory_examples(trajectories, title, n_examples=6, seed=SEED + 17):
    rng_examples = np.random.default_rng(seed)
    if len(trajectories) <= n_examples:
        examples = list(trajectories)
    else:
        indices = rng_examples.choice(len(trajectories), size=n_examples, replace=False)
        examples = [trajectories[int(index)] for index in indices]

    cols = min(3, len(examples))
    rows = int(math.ceil(len(examples) / cols))
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[
            f"step={t.step_size:g}, steps={t.step_count}, repeat={t.repeat}"
            for t in examples
        ],
        horizontal_spacing=0.13,
        vertical_spacing=0.16,
    )

    for index, example in enumerate(examples):
        row = index // cols + 1
        col = index % cols + 1
        for segment_lons, segment_ys in wrap_aware_segments(
            example.positions_lon, example.positions_y
        ):
            fig.add_trace(
                go.Scatter(
                    x=segment_lons,
                    y=segment_ys,
                    mode="lines+markers",
                    line={"width": 2},
                    marker={"size": 6},
                    showlegend=False,
                ),
                row=row,
                col=col,
            )
        fig.add_trace(
            go.Scatter(
                x=[example.start_lon],
                y=[example.start_y],
                mode="markers",
                marker={"symbol": "star", "size": 12, "color": "red"},
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(range=[-np.pi, np.pi], title_text="x = lon", row=row, col=col)
        fig.update_yaxes(
            range=[-1.0, 1.0], title_text="y", title_standoff=10, row=row, col=col
        )

    fig.update_layout(
        title={"text": title, "x": 0.5},
        height=350 * rows,
        width=1250,
        margin={"l": 75, "r": 45, "t": 90, "b": 75},
    )
    fig.show()

## 3. Biased Direction Random Walk

This section investigates a single biased-direction random walk. Headings are drawn from a von Mises distribution around `DIRECTION_HEADING`; `DIRECTION_MAGNITUDE` controls how strongly headings concentrate around that direction.

In [7]:
direction_trajectories = make_trajectories(rng)

print(f"biased-direction trajectories: {len(direction_trajectories)}")
print(f"DIRECTION_MAGNITUDE: {DIRECTION_MAGNITUDE:g}")
print(f"DIRECTION_HEADING: {DIRECTION_HEADING:g}")

biased-direction trajectories: 420
DIRECTION_MAGNITUDE: 0.75
DIRECTION_HEADING: 0


In [8]:
plot_trajectory_examples(
    direction_trajectories,
    "biased-direction random trajectory examples",
    n_examples=6,
)
direction_information_rows = estimate_information(
    direction_trajectories, progress_label="biased-direction trajectory"
)
print_information_summary(
    direction_information_rows, "Biased-direction information summary"
)

mean, sem, count = summarize_values(
    mi_values(direction_information_rows, lambda row: True)
)
print(
    f"\nDIRECTION_MAGNITUDE={DIRECTION_MAGNITUDE:g}: "
    f"{mean:6.3f} +/- {sem:5.3f} bits over {count:3d} trajectories"
)

plot_aggregate_mi(direction_information_rows, "biased direction")
plot_mi_heatmap(direction_information_rows, "biased direction MI heatmap")

estimating biased-direction trajectory: 100%|██████████| 420/420 [07:03<00:00,  1.01s/traj]


Biased-direction information summary
prior entropy:                   14.651 bits
mean H(start | trajectory):       3.478 bits
median H(start | trajectory):     3.281 bits
mean I(start; trajectory):       11.173 bits
median I(start; trajectory):     11.370 bits
median MAP angular error:        0.0224 rad

DIRECTION_MAGNITUDE=0.75: 11.173 +/- 0.082 bits over 420 trajectories
